# 01 — Dataset Understanding

**Sprint 4: Data Inspection & Exploratory Data Analysis (EDA)**

Understand the dataset's structure, identify its features and target, classify columns by their roles and data types, and document its dimensions before performing data-quality checks or preprocessing.

## 1. What is a Dataset?

### Markdown — Concept
A **dataset** is an organized collection of related observations. It is commonly represented as a table: each **row** represents one observation, and each **column** represents an attribute recorded about that observation.

### Example
A telecom company may store one row per customer, with columns for tenure, monthly charges, contract type, and whether the customer churned.

### Code
Load the data and display a few records.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# If this notebook is in the project root rather than a notebooks/ folder,
# use: Path("dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_PATH.exists():
    alternative = Path("dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    if alternative.exists():
        DATA_PATH = alternative

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Download the CSV and update DATA_PATH to its location."
    )

df = pd.read_csv(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Output explanation
The table shows the first five customer records. Each row is an observation, and each column is a recorded customer attribute.
Looking at sample rows helps us understand what one record represents and whether the column names and values look sensible.

### AI/ML relevance
The dataset is the raw material for analysis and model development. Understanding its unit of observation helps prevent mistakes such as treating multiple records for one customer as independent observations.

## 2. What is a Feature?

### Markdown — Concept
A **feature** is an input attribute that describes an observation. In a customer churn problem, examples include `tenure`, `MonthlyCharges`, `Contract`, and `InternetService`.

A feature may be numerical, categorical, date/time, text, or another data type. A column is not automatically useful just because it exists; its meaning and quality must be examined.

### Example
`tenure = 24` describes how many months a customer has stayed with the company. `Contract = "Month-to-month"` describes the customer's contract category.

### Code
List all columns and display a few values from each.

In [5]:
print("Columns in the dataset:")
for column in df.columns:
    print(f"- {column}")

print("\nExample values:")
df.head(3).T

Columns in the dataset:
- customerID
- gender
- SeniorCitizen
- Partner
- Dependents
- tenure
- PhoneService
- MultipleLines
- InternetService
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
- Contract
- PaperlessBilling
- PaymentMethod
- MonthlyCharges
- TotalCharges
- Churn

Example values:


,0,1,2
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK
gender,Female,Male,Male
SeniorCitizen,0,0,0
Partner,Yes,No,No
Dependents,No,No,No
tenure,1,34,2
PhoneService,No,Yes,Yes
MultipleLines,No phone service,No,No
InternetService,DSL,DSL,DSL
OnlineSecurity,No,Yes,Yes


### Output explanation
The column list names the available attributes. The transposed preview displays columns as rows, making it easier to scan each feature and its example values. Features may describe customer profile, subscribed services, account settings, or billing. These groups can help organize later analyses.

### AI/ML relevance
Features may become model inputs after suitable validation and preprocessing. Identifier columns and information unavailable at prediction time may need to be excluded.

## 3. What is a Target?

### Markdown — Concept
The **target** is the outcome a supervised machine-learning model is designed to predict. It is also called the label or dependent variable.

### Example
For churn prediction, `Churn` is the target. Values such as `Yes` and `No` indicate whether a customer left the company.

### Code
Check whether the expected target column exists and inspect its values.

In [6]:
TARGET_COLUMN = "Churn"

if TARGET_COLUMN in df.columns:
    print("Target column:", TARGET_COLUMN)
    print(df[TARGET_COLUMN].value_counts(dropna=False))
else:
    print(f"{TARGET_COLUMN!r} was not found. Check the dataset's column names.")

Target column: Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64


### Output explanation
The output lists the observed target values and their counts, including missing values if any. The target distribution shows how many records belong to each outcome. This is an initial check; class balance and target quality will be investigated more deeply in the target-analysis notebook.

### AI/ML relevance
A model learns patterns in the input features to predict the target. The target must be defined correctly, and target information must not accidentally leak into the input features.

## 4. Independent Variables

### Markdown — Concept
**Independent variables** are the input variables used to explain or predict an outcome. In machine learning, they are commonly represented by `X`.

The word *independent* does not guarantee that the variables are statistically independent of one another.

### Example
`tenure`, `MonthlyCharges`, and `Contract` may be used as inputs to predict churn.

### Code
Create a preliminary feature list by excluding the target and the customer identifier.

In [7]:
identifier_candidates = ["customerID"]
excluded_from_X = [TARGET_COLUMN] + [
    col for col in identifier_candidates if col in df.columns
]
X_columns = [col for col in df.columns if col not in excluded_from_X]

print("Preliminary input columns (X):")
print(X_columns)
print("\nNumber of preliminary input columns:", len(X_columns))

Preliminary input columns (X):
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Number of preliminary input columns: 19


### Output explanation
The output lists columns that could be considered inputs for a first churn model, excluding `Churn` and the obvious customer ID.
This is only a preliminary selection. Some columns may later be removed or transformed based on data quality, business meaning, leakage risk, and model requirements.

### AI/ML relevance
Separating inputs from the target makes the prediction task explicit and helps avoid accidentally training the model using the answer it is supposed to predict.

## 5. Dependent Variables

### Markdown — Concept
The **dependent variable** is the outcome being studied or predicted. In supervised learning, it is the target `y`. Its value may depend on several inputs, although a relationship in the data does not by itself prove causation.

### Example
A telecom company wants to predict whether a customer will churn. The target is `Churn`, while contract type, tenure, and monthly charges are candidate inputs.

### Code
Separate the target from the preliminary inputs.

In [8]:
if TARGET_COLUMN in df.columns:
    y = df[TARGET_COLUMN]
    X = df[X_columns]
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    display(pd.DataFrame({"target_example": y.head()}))
else:
    print("Set TARGET_COLUMN to the correct target column before creating X and y.")

X shape: (7043, 19)
y shape: (7043,)


,target_example
0,No
1,No
2,Yes
3,No
4,Yes


### Output explanation
`X` contains the preliminary input columns, while `y` contains the target column. The row counts should match because each input record corresponds to one target value.
Keeping `X` and `y` separate makes the prediction problem easier to reason about and validate.

### AI/ML relevance
Many supervised-learning APIs expect input features and target labels as separate objects. A later train/test split must be performed carefully so that evaluation reflects unseen records.

## 6. Numerical Features

### Markdown — Concept
**Numerical features** contain values that represent quantities or counts and can be analyzed using mathematical operations. Examples include `tenure`, `MonthlyCharges`, and `SeniorCitizen` in this dataset.

Be careful: a column stored as numbers is not necessarily a meaningful quantity. For example, an encoded category such as 0/1 may represent a group rather than a measured amount.

### Example
`MonthlyCharges = 75.50` is a monetary amount. `tenure = 12` represents the number of months a customer has stayed.

### Code
Identify numeric columns from their pandas data types.

In [9]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
print("Numerical columns:")
print(numeric_columns)

if numeric_columns:
    display(df[numeric_columns].head())

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges']


,SeniorCitizen,tenure,MonthlyCharges
0,0,1,29.85
1,0,34,56.95
2,0,2,53.85
3,0,45,42.30
4,0,2,70.70


### Output explanation
Pandas lists columns whose current data types are numeric, such as integers and floating-point numbers.
Check the meaning of each column before interpreting it numerically. In this dataset, `SeniorCitizen` is often encoded as 0/1 and is better understood as a binary category for many business analyses.

### AI/ML relevance
Numeric features may need scaling, transformation, or missing-value treatment depending on the model. Their type and meaning should be confirmed before choosing preprocessing methods.

## 7. Categorical Features

### Markdown — Concept
**Categorical features** represent groups or labels rather than measured quantities. Examples include `Contract`, `PaymentMethod`, `InternetService`, and `Partner`.

Categories can be **nominal** (no natural order, such as payment method) or **ordinal** (a meaningful order, such as low, medium, high).

### Example
`Contract = "Two year"` is a category. It should not be treated as the number 2 simply because its label contains a number.

### Code
Identify columns stored as object or category data types.

In [10]:
categorical_columns = df.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Categorical/text columns based on current dtype:")
print(categorical_columns)

for column in categorical_columns[:5]:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False).head(10))

Categorical/text columns based on current dtype:
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']

customerID:
customerID
7590-VHVEG    1
5575-GNVDE    1
3668-QPYBK    1
7795-CFOCW    1
9237-HQITU    1
9305-CDSKC    1
1452-KIOVK    1
6713-OKOMC    1
7892-POOKP    1
6388-TABGU    1
Name: count, dtype: int64

gender:
gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner:
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents:
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService:
PhoneService
Yes    6361
No      682
Name: count, dtype: int64


### Output explanation
The output lists text/category columns and shows the most frequent values for up to five of them.
Frequency tables reveal common labels and can expose inconsistent spelling, unexpected values, or rare categories. Some object columns may contain identifiers or numeric-looking text rather than true categories.

### AI/ML relevance
Many models require categorical values to be encoded into numeric representations. The encoding method depends on the meaning and number of categories.

## 8. Discrete Features

### Markdown — Concept
A **discrete feature** takes separate, countable values, often integers. Examples include the number of products purchased or the number of service tickets raised.

Discrete values can sometimes be stored as floating-point numbers, so dtype alone does not establish whether a feature is discrete.

### Example
`tenure` is recorded as whole months in this dataset, making it count-like. `SeniorCitizen` is binary-coded and represents membership in a category.

### Code
Inspect the values and number of unique values in selected count-like columns.

In [11]:
discrete_candidates = [
    col for col in ["tenure", "SeniorCitizen"]
    if col in df.columns
]

for column in discrete_candidates:
    print(f"\n{column}")
    print("Unique values (first 20):", df[column].dropna().unique()[:20])
    print("Number of unique values:", df[column].nunique(dropna=True))


tenure
Unique values (first 20): [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30]
Number of unique values: 73

SeniorCitizen
Unique values (first 20): [0 1]
Number of unique values: 2


### Output explanation
The output shows example values and the number of distinct non-missing values in the selected columns.
`tenure` represents whole months, while `SeniorCitizen` is a binary indicator. The latter is categorical in meaning despite its numeric storage.

### AI/ML relevance
Discrete count features may be summarized differently from continuous measurements. Confirming meaning helps avoid inappropriate scaling, plots, or statistical interpretations.

## 9. Continuous Features

### Markdown — Concept
A **continuous feature** represents a measurement that can, in principle, take any value within a range. Examples include time, distance, temperature, and monetary measurements recorded with decimals.

Real datasets often round continuous measurements, so their stored values may look discrete.

### Example
`MonthlyCharges` and `TotalCharges` are monetary quantities. `TotalCharges` may be stored as text in some dataset versions because blank entries interfere with numeric parsing.

### Code
Inspect selected charge columns and their current types.

In [12]:
continuous_candidates = [
    col for col in ["MonthlyCharges", "TotalCharges"]
    if col in df.columns
]

for column in continuous_candidates:
    print(f"\n{column}")
    print("Current dtype:", df[column].dtype)
    print(df[column].head(10).to_list())


MonthlyCharges
Current dtype: float64
[29.85, 56.95, 53.85, 42.3, 70.7, 99.65, 89.1, 29.75, 104.8, 56.15]

TotalCharges
Current dtype: str
['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95']


### Output explanation
The output shows each selected column's current dtype and example values. A numeric-looking column stored as `object` may contain blanks or other non-numeric strings.
Do not assume that every value in a charge column is valid numeric data. Investigate the content and record any conversion issues in the data-quality notebook rather than silently changing the original data here.

### AI/ML relevance
A model may fail or behave incorrectly if a measurement is stored as text. Later preprocessing can convert validated values to numeric form and decide how to handle invalid entries.

## 10. Identifier Columns

### Markdown — Concept
An **identifier column** distinguishes an entity or record, such as a customer ID, order ID, or transaction ID. It often helps with tracing records but usually does not describe a useful generalizable property of the entity.

### Example
`customerID` identifies a customer. The string itself should not normally be interpreted as a measure of customer behavior.

### Code
Inspect likely identifier columns.

In [17]:
identifier_columns = [
    col for col in df.columns
    if col.lower() in {"customerid", "customer_id", "id", "record_id"}
    or col.lower().endswith("_id")
]

print("Potential identifier columns:", identifier_columns)
for column in identifier_columns:
    print(f"\n{column}")
    print("Unique values:", df[column].nunique(dropna=False))
    display(df[[column]].head())

Potential identifier columns: ['customerID']

customerID
Unique values: 7043


,customerID
0,7590-VHVEG
1,5575-GNVDE
2,3668-QPYBK
3,7795-CFOCW
4,9237-HQITU


### Output explanation
The output identifies likely ID columns and reports their distinct-value counts.
If nearly every row has a different ID, the column likely functions as an identifier. Confirm this using the dataset documentation and business context.

### AI/ML relevance
High-cardinality identifiers can cause overfitting or create meaningless patterns. Retain them for record tracking when needed, but evaluate carefully before using them as model features.

## 11. Target Variable

### Markdown — Concept
A **target variable** is the specific outcome that a supervised model predicts. A classification target contains classes; a regression target contains a numeric quantity.

A dataset can have no target if it is intended for unsupervised analysis, or it may contain multiple outcomes for different tasks.

### Example
For this churn dataset, `Churn` is a binary classification target with the expected labels `Yes` and `No`.

### Code
Inspect target values and check for unexpected labels.

In [18]:
if TARGET_COLUMN in df.columns:
    target_values = df[TARGET_COLUMN].value_counts(dropna=False)
    print("Target distribution:")
    display(target_values.to_frame("count"))

    expected_labels = {"Yes", "No"}
    observed_labels = set(df[TARGET_COLUMN].dropna().astype(str).unique())
    print("Observed non-missing labels:", observed_labels)
    print("Unexpected labels:", observed_labels - expected_labels)
else:
    print("Target column not found. Update TARGET_COLUMN.")

Target distribution:


,count
Churn,
No,5174
Yes,1869


Observed non-missing labels: {'Yes', 'No'}
Unexpected labels: set()


### Output explanation
The counts show the number of records per target value. The unexpected-label check compares observed labels against the expected `Yes` and `No` values.
Unexpected labels or missing target values should be investigated. This cell does not modify or remove any records.

### AI/ML relevance
Target errors directly affect what a model learns. Class proportions also influence the choice of evaluation metrics and whether class imbalance needs special attention.

## 12. Metadata

### Markdown — Concept
**Metadata** is information that describes a dataset or its columns rather than being the main observation itself. It may include the source, collection date, units, definitions, data owner, license, and update frequency.

In a CSV, some metadata may be documented outside the file, so the dataset's source page and data dictionary are important.

### Example
For this dataset, useful metadata includes its telecommunications domain, the meaning of `Churn`, the unit of `tenure` (months), and the source URL.

### Code
Record basic dataset documentation in a small metadata dictionary.

In [19]:
dataset_metadata = {
    "dataset_name": "IBM Telco Customer Churn",
    "source": "https://www.kaggle.com/datasets/blastchar/telco-customer-churn",
    "domain": "Telecommunications / Customer Retention",
    "unit_of_observation": "One customer record",
    "target_column": TARGET_COLUMN,
    "target_task": "Binary classification",
    "notes": (
        "Confirm column definitions and any dataset-specific caveats against "
        "the source documentation and the exact CSV version used."
    ),
}

for key, value in dataset_metadata.items():
    print(f"{key}: {value}")

dataset_name: IBM Telco Customer Churn
source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
domain: Telecommunications / Customer Retention
unit_of_observation: One customer record
target_column: Churn
target_task: Binary classification
notes: Confirm column definitions and any dataset-specific caveats against the source documentation and the exact CSV version used.


### Output explanation
The output displays the documentation fields recorded for the project.
Metadata makes analysis reproducible and helps reviewers understand where the data came from, what one row means, and how the target is defined.

### AI/ML relevance
Source, units, collection context, and target definitions help identify leakage risks, limitations, and whether the data is appropriate for the intended prediction task.

## 13. Dataset Dimensions

### Markdown — Concept
**Dataset dimensions** describe the size of a tabular dataset: the number of rows and columns. In pandas, `shape` returns a tuple in the order `(rows, columns)`.

### Example
A dataset with 7,043 rows and 21 columns has 7,043 observations and 21 recorded attributes. Your actual output depends on the exact CSV file.

### Code
Check the actual dimensions and basic structural counts.

In [17]:
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)
print("Shape:", df.shape)
print("Number of duplicate rows:", df.duplicated().sum())
print("Memory usage (bytes):", df.memory_usage(deep=True).sum())

Number of rows: 7043
Number of columns: 21
Shape: (7043, 21)
Number of duplicate rows: 0
Memory usage (bytes): 1954553


### Output explanation
`shape` reports the number of observations and columns. The duplicate count is a preliminary structural check; duplicate analysis will be covered in depth in Notebook 03. Memory usage estimates the DataFrame's in-memory footprint.
Dimensions provide a baseline for checking whether the file loaded as expected. A surprising row or column count may indicate a wrong file, parsing issue, or different dataset version.

### AI/ML relevance
Dataset size affects computational cost and evaluation strategy. The number of rows alone does not guarantee data quality or representativeness.